In [1]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Generate 100 test samples
num_samples = 100

test_data = {
    "km_driven": np.random.randint(1000, 150000, num_samples),
    "mileage_new": np.round(np.random.uniform(10, 25, num_samples), 1),  # kmpl
    "max_power": np.random.uniform(50, 250, num_samples),  # Raw power (bhp)
    "max_torque": np.random.uniform(100, 400, num_samples),  # Raw torque (Nm)
    "max_engine_capacity_new": np.random.randint(1000, 3000, num_samples),  # cc
}

# Apply log transforms (as done in training)
test_data["log_max_power"] = np.log(test_data["max_power"])
test_data["log_max_torque"] = np.log(test_data["max_torque"])
test_data["log_max_engine_capacity_new"] = np.log(test_data["max_engine_capacity_new"])

# Drop raw columns (keep only log-transformed features)
test_data.pop("max_power")
test_data.pop("max_torque")
test_data.pop("max_engine_capacity_new")

# Add synthetic "price" (for validation)
test_data["price"] = (
    test_data["km_driven"] * -0.2 +
    test_data["mileage_new"] * 5000 +
    np.exp(test_data["log_max_power"]) * 10000 +
    np.random.normal(0, 200000, num_samples)  # Noise
).astype(int)

# Create DataFrame
test_df = pd.DataFrame(test_data)

# Save to CSV
test_df.to_csv("unseen_car_price_test_data.csv", index=False)

PermissionError: [Errno 13] Permission denied: 'unseen_car_price_test_data.csv'

In [2]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
 

In [3]:
test_df = pd.read_csv("unseen_car_price_test_data.csv")
X_test = test_df.drop("price", axis=1)
X_test = scaler.fit_transform(X_test)
y_test = test_df["price"]

In [4]:
from tensorflow import keras 
loaded_model = keras.models.load_model('trained_model.keras')
loaded_model.summary()

Model: "sequential_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_88 (Dense)                │ (None, 128)            │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_32 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_89 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_33 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_90 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_34 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_91 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,100 (266.02 KB)

 Trainable params: 34,049 (133.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 34,051 (133.02 KB)

In [5]:
predictions = loaded_model.predict(X_test)

ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "dense_88" is incompatible with the layer: expected axis -1 of input shape to have value 6, but received input with shape (32, 5)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(32, 5), dtype=float32)
  • training=False
  • mask=None

In [ ]:
 test_loss, test_mae = loaded_model.evaluate(X_test,y_test)
 print(f'Mean Absolute Error (MAE) on the test set: {test_mae:.4f}')